[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/data_analysis_assistant/crewai.ipynb)

# Data-analysis assistant — CrewAI Flow

Inspector, analyst, executor and reporter are explicit Flow stages over the matched dataset and oracle. Mock runtime is about one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import contextlib
import csv
import io
import json
import os
import tempfile
from pathlib import Path
from typing import ClassVar

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai.flow.flow import Flow, listen, start  # noqa: E402
from pydantic import BaseModel, ConfigDict  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture = json.loads((ROOT / "data/data_analysis_assistant/case_mock.json").read_text())

## Flow
The analyst selects a typed allowlisted operation; the executor—not the model—computes and validates values. Unsupported tools, schemas or results stop before reporting.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Decision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: str
    group_column: str
    before_column: str
    after_column: str


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


def fresh_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


class AnalysisFlow(Flow):
    @start()
    def inspector(self):
        self.client = fresh_model()
        handle = data_path.open(newline="", encoding="utf-8")
        rows = list(csv.DictReader(handle))
        handle.close()
        return {
            "rows": len(rows),
            "columns": tuple(rows[0]),
            "trace": [{"event": "inspect"}],
            "model_calls": 0,
        }

    @listen(inspector)
    async def analyst(self, state):
        decision = await propose(
            self.client, Decision, f"Choose permitted analysis for {state['columns']}"
        )
        return {
            **state,
            "decision": decision,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "decision"}],
        }

    @listen(analyst)
    def executor(self, state):
        if state["decision"].tool != "mean_reduction_by_intervention":
            return {**state, "valid": False, "outcome": "fallback"}
        request = AnalysisRequest(
            group_column=state["decision"].group_column,
            before_column=state["decision"].before_column,
            after_column=state["decision"].after_column,
        )
        result = summarise_reduction(data_path, request)
        valid = result == expected["mean_reduction_kg"]
        return {
            **state,
            "result": result,
            "valid": valid,
            "trace": [*state["trace"], {"event": "execute_and_validate", "valid": valid}],
        }

    @listen(executor)
    async def reporter(self, state):
        if not state["valid"]:
            return {**state, "outcome": "fallback", "termination": "validation_failed"}
        report = await propose(self.client, Report, f"Report only {state['result']}")
        valid = set(report.groups) == set(state["result"])
        return {
            **state,
            "report": report,
            "outcome": "supported_report" if valid else "fallback",
            "model_calls": 2,
            "termination": "criteria_met" if valid else "validation_failed",
            "trace": [*state["trace"], {"event": "report_validation", "valid": valid}],
        }


async def run_flow():
    with contextlib.redirect_stdout(io.StringIO()):
        return await AnalysisFlow().kickoff_async()


first = await run_flow()
second = await run_flow()
try:
    AnalysisRequest(group_column="missing", before_column="before_kg", after_column="after_kg")
    schema_mismatch_rejected = False
except ValueError:
    schema_mismatch_rejected = True
failure_demonstrations = {
    "invalid_tool": "executor returns fallback",
    "unsafe_code": "never evaluated",
    "schema_mismatch": schema_mismatch_rejected,
    "result_mismatch": "reporter returns fallback",
}
evaluation = {
    "component": first["result"] == expected["mean_reduction_kg"],
    "trajectory": len(first["trace"]) == 4 and first["model_calls"] <= 2,
    "task": first["outcome"] == "supported_report",
    "safety": first["decision"].tool == "mean_reduction_by_intervention"
    and schema_mismatch_rejected,
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "flow_stages": 4},
    "failure_demonstrations": failure_demonstrations,
    "provenance": file_sha256(data_path),
    "fallback": "invalid tools or outputs stop reporting",
}

## Limitation
CrewAI role separation adds overhead but no causal validity; arbitrary code remains intentionally excluded.